# Notebook 03 — Financial Stability Index & MPC Policy Stance
Builds two indices:
- **FSSI** (Financial Stability Sentiment Index) — from FSR reports, used as an *outcome variable*
- **MPSI** (Monetary Policy Stance Index) — from MPC minutes, the core independent variable
- Classifies each MPC period as **Hawkish / Dovish / Neutral**

In [1]:
import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_PROC = os.path.join(PROJECT_ROOT, 'data', 'processed')
OUTPUT_RES = os.path.join(PROJECT_ROOT, 'output', 'results')
os.makedirs(OUTPUT_RES, exist_ok=True)

# Load combined sentiment (from Notebook 02)
sent_path = os.path.join(DATA_PROC, 'combined_sentiment_lm.csv')
if not os.path.exists(sent_path):
    sent_path = os.path.join(DATA_PROC, 'combined_sentiment_aligned.csv')
    
df = pd.read_csv(sent_path)
print(f"Loaded sentiment data: {df.shape}")
print(df.columns.tolist())
print(df.head())

Loaded sentiment data: (31, 11)
['Period', 'MPC_Sentiment', 'MPC_Polarity', 'MPC_Positive', 'MPC_Negative', 'MPC_TotalWords', 'FSR_Sentiment', 'FSR_Polarity', 'FSR_Positive', 'FSR_Negative', 'FSR_TotalWords']
   Period  MPC_Sentiment  MPC_Polarity  MPC_Positive  MPC_Negative  \
0  Jul-10       0.001410           1.0            40            33   
1  Oct-10       0.001129           1.0            63            49   
2  Jul-11      -0.002446          -1.0           101           170   
3  Oct-11      -0.000499          -1.0            58            64   
4  Jul-12      -0.010971          -1.0            15            63   

   MPC_TotalWords  FSR_Sentiment  FSR_Polarity  FSR_Positive  FSR_Negative  \
0            4964      -0.003694          -1.0           302           495   
1           12395      -0.003915          -1.0           354           578   
2           28214      -0.005365          -1.0           216           468   
3           12018      -0.007830          -1.0           1

## Standardise date column

In [2]:
# Handle both column name formats
if 'Period' in df.columns:
    df = df.rename(columns={'Period':'Date'})
elif 'MPC_Date' in df.columns:
    df = df.rename(columns={'MPC_Date':'Date', 'MPC_Sentiment':'MPC_Sentiment', 'FSR_Sentiment':'FSR_Sentiment'})

date_map = {
    'Jul-10':'2010-07','Oct-10':'2010-10','Jul-11':'2011-07','Oct-11':'2011-10',
    'Jul-12':'2012-07','Oct-12':'2012-10','Jul-13':'2013-07','Oct-13':'2013-10',
    'Jun-14':'2014-06','Dec-14':'2014-12','Jun-15':'2015-06','Dec-15':'2015-12',
    'Jun-16':'2016-06','Dec-16':'2016-12','Jun-17':'2017-06','Dec-17':'2017-12',
    'Jun-18':'2018-06','Dec-18':'2018-12','Jun-19':'2019-06','Dec-19':'2019-12',
    'Jun-20':'2020-06','Dec-20':'2020-12','Jun-21':'2021-06','Dec-21':'2021-12',
    'Jun-22':'2022-06','Dec-22':'2022-12','Jun-23':'2023-06','Dec-23':'2023-12',
    'Jun-24':'2024-06','Dec-24':'2024-12','Jun-25':'2025-06',
    'Jul-2010':'2010-07','Oct-2010':'2010-10','Jul-2011':'2011-07','Oct-2011':'2011-10',
}
df['Date'] = df['Date'].map(lambda x: date_map.get(str(x), str(x)))
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m', errors='coerce')
df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Records: {len(df)}")

Date range: 2010-07-01 00:00:00 to 2025-06-01 00:00:00
Records: 31


## Build FSSI — Financial Stability Sentiment Index
FSR sentiment is an *outcome variable* in this study — it measures how monetary policy transmission affects financial system stability.

In [3]:
# Invert FSR sentiment: more negative FSR tone = lower financial stability
df['FSSI_raw'] = -1 * df['FSR_Sentiment']

# Min-max normalise to 0–100 scale
min_v = df['FSSI_raw'].min()
max_v = df['FSSI_raw'].max()
df['FSSI'] = 100 * (df['FSSI_raw'] - min_v) / (max_v - min_v)

# 3-period rolling average to smooth short-term noise
df['FSSI_smoothed'] = df['FSSI'].rolling(window=3, min_periods=1).mean().round(2)

print("FSSI constructed:")
print(df[['Date','FSR_Sentiment','FSSI','FSSI_smoothed']].to_string(index=False))

FSSI constructed:
      Date  FSR_Sentiment       FSSI  FSSI_smoothed
2010-07-01      -0.003694   0.000000           0.00
2010-10-01      -0.003915   4.570958           2.29
2011-07-01      -0.005365  34.551925          13.04
2011-10-01      -0.007830  85.510325          41.54
2012-07-01      -0.008530 100.000000          73.35
2012-10-01      -0.008051  90.080246          91.86
2013-07-01      -0.005977  47.203491          79.09
2013-10-01      -0.007977  88.550338          75.28
2014-06-01      -0.005361  34.455315          56.74
2014-12-01      -0.003993   6.177526          43.06
2015-06-01      -0.004150   9.429607          16.69
2015-12-01      -0.007503  78.749481          31.45
2016-06-01      -0.005397  35.217677          41.13
2016-12-01      -0.005058  28.197813          47.39
2017-06-01      -0.003734   0.826176          21.41
2017-12-01      -0.004733  21.479391          16.83
2018-06-01      -0.006149  50.753912          24.35
2018-12-01      -0.004963  26.237103          

## Build MPSI — Monetary Policy Stance Index

In [4]:
# Normalise MPC sentiment to 0–100 (positive = accommodative, negative = restrictive)
mpc_min = df['MPC_Sentiment'].min()
mpc_max = df['MPC_Sentiment'].max()
df['MPSI'] = 100 * (df['MPC_Sentiment'] - mpc_min) / (mpc_max - mpc_min)
df['MPSI_smoothed'] = df['MPSI'].rolling(window=3, min_periods=1).mean().round(2)

print("MPSI (Monetary Policy Stance Index) constructed.")
print(df[['Date','MPC_Sentiment','MPSI','MPSI_smoothed']].to_string(index=False))

MPSI (Monetary Policy Stance Index) constructed.
      Date  MPC_Sentiment       MPSI  MPSI_smoothed
2010-07-01       0.001410  81.501740          81.50
2010-10-01       0.001129  79.654260          80.58
2011-07-01      -0.002446  56.121289          72.43
2011-10-01      -0.000499  68.933090          68.24
2012-07-01      -0.010971   0.000000          41.68
2012-10-01      -0.002900  53.131568          40.69
2013-07-01      -0.005913  33.299754          28.81
2013-10-01       0.000780  77.355299          54.60
2014-06-01       0.000000  72.219410          60.96
2014-12-01      -0.003920  46.416934          65.33
2015-06-01      -0.009314  10.909095          43.18
2015-12-01      -0.002489  55.836885          37.72
2016-06-01      -0.000442  69.309375          45.35
2016-12-01      -0.006456  29.725977          51.62
2017-06-01      -0.002428  56.236347          51.76
2017-12-01       0.000299  74.187566          53.38
2018-06-01       0.000871  77.949965          69.46
2018-12-01     

## Hawkish / Dovish / Neutral Classification
Classify each MPC period based on sentiment deviation from the mean — a standard approach in central bank communication research.

In [5]:
# RBI actual repo rate decisions (manually verified from RBI records)
REPO_RATES = {
    '2010-07':5.75,'2010-10':6.25,'2011-07':7.50,'2011-10':8.50,
    '2012-07':8.00,'2012-10':8.00,'2013-07':7.25,'2013-10':7.75,
    '2014-06':8.00,'2014-12':8.00,'2015-06':7.25,'2015-12':6.75,
    '2016-06':6.50,'2016-12':6.25,'2017-06':6.25,'2017-12':6.00,
    '2018-06':6.25,'2018-12':6.50,'2019-06':5.75,'2019-12':5.15,
    '2020-06':4.00,'2020-12':4.00,'2021-06':4.00,'2021-12':4.00,
    '2022-06':4.90,'2022-12':6.25,'2023-06':6.50,'2023-12':6.50,
    '2024-06':6.50,'2024-12':6.25,'2025-06':6.00,
}
df['Repo_Rate'] = df['Date'].dt.strftime('%Y-%m').map(REPO_RATES)
df['Repo_Change'] = df['Repo_Rate'].diff()

# Sentiment-based stance classification
mean_si = df['MPC_Sentiment'].mean()
std_si  = df['MPC_Sentiment'].std()

def classify_stance(si):
    if si > mean_si + 0.3 * std_si:
        return 'Dovish'
    elif si < mean_si - 0.3 * std_si:
        return 'Hawkish'
    else:
        return 'Neutral'

df['Stance'] = df['MPC_Sentiment'].apply(classify_stance)

stance_counts = df['Stance'].value_counts()
print("Stance distribution:")
print(stance_counts)
print()
print(df[['Date','MPC_Sentiment','Stance','Repo_Rate','Repo_Change']].to_string(index=False))

Stance distribution:
Stance
Dovish     15
Hawkish    10
Neutral     6
Name: count, dtype: int64

      Date  MPC_Sentiment  Stance  Repo_Rate  Repo_Change
2010-07-01       0.001410  Dovish       5.75          NaN
2010-10-01       0.001129  Dovish       6.25         0.50
2011-07-01      -0.002446 Neutral       7.50         1.25
2011-10-01      -0.000499  Dovish       8.50         1.00
2012-07-01      -0.010971 Hawkish       8.00        -0.50
2012-10-01      -0.002900 Neutral       8.00         0.00
2013-07-01      -0.005913 Hawkish       7.25        -0.75
2013-10-01       0.000780  Dovish       7.75         0.50
2014-06-01       0.000000  Dovish       8.00         0.25
2014-12-01      -0.003920 Hawkish       8.00         0.00
2015-06-01      -0.009314 Hawkish       7.25        -0.75
2015-12-01      -0.002489 Neutral       6.75        -0.50
2016-06-01      -0.000442  Dovish       6.50        -0.25
2016-12-01      -0.006456 Hawkish       6.25        -0.25
2017-06-01      -0.002428 Neutral

## Mark structural break events

In [6]:
KEY_EVENTS = {
    '2013-07': 'Taper Tantrum',
    '2016-10': 'MPC Framework Start',
    '2016-12': 'Demonetisation',
    '2019-06': 'Pre-COVID Rate Cut Cycle',
    '2020-06': 'COVID Emergency Cut',
    '2022-06': 'Post-COVID Rate Hike Cycle',
    '2023-06': 'Pause at Peak Rate',
    '2024-12': 'Rate Cut Restart',
}
df['Event'] = df['Date'].dt.strftime('%Y-%m').map(KEY_EVENTS).fillna('')

event_rows = df[df['Event'] != ''][['Date','Stance','Repo_Rate','Event']]
print("Key structural events in sample:")
print(event_rows.to_string(index=False))

Key structural events in sample:
      Date  Stance  Repo_Rate                      Event
2013-07-01 Hawkish       7.25              Taper Tantrum
2016-12-01 Hawkish       6.25             Demonetisation
2019-06-01 Hawkish       5.75   Pre-COVID Rate Cut Cycle
2020-06-01 Hawkish       4.00        COVID Emergency Cut
2022-06-01 Neutral       4.90 Post-COVID Rate Hike Cycle
2023-06-01  Dovish       6.50         Pause at Peak Rate
2024-12-01 Hawkish       6.25           Rate Cut Restart


In [7]:
# Save final index file
index_path = os.path.join(DATA_PROC, 'policy_indices.csv')
df.to_csv(index_path, index=False)
print(f"Policy indices saved: {index_path}")
print(df.columns.tolist())

print("\n" + "="*50)
print("NOTEBOOK 03 COMPLETE")
print("="*50)

Policy indices saved: C:\Users\rohit\Desktop\Rohit\RBI_Sentiment_Project\data\processed\policy_indices.csv
['Date', 'MPC_Sentiment', 'MPC_Polarity', 'MPC_Positive', 'MPC_Negative', 'MPC_TotalWords', 'FSR_Sentiment', 'FSR_Polarity', 'FSR_Positive', 'FSR_Negative', 'FSR_TotalWords', 'FSSI_raw', 'FSSI', 'FSSI_smoothed', 'MPSI', 'MPSI_smoothed', 'Repo_Rate', 'Repo_Change', 'Stance', 'Event']

NOTEBOOK 03 COMPLETE
